In [2]:
import os
from PIL import Image
import numpy as np
from collections import Counter

# Ajuste para o seu diretório de dataset
DATASET_DIR = './dataset_organizado'
IMG_EXTS = ('.png', '.jpg', '.jpeg', '.bmp', '.tiff')

def check_image(path, expected_size=(224, 224)):
    try:
        img = Image.open(path)
        img = img.convert('RGB')
        arr = np.array(img)
        # Check for NaN or Inf
        if np.isnan(arr).any() or np.isinf(arr).any():
            print(f"[ERRO] NaN ou Inf encontrado em: {path}")
            return False
        # Check size
        if img.size != expected_size:
            print(f"[AVISO] Tamanho inesperado {img.size} em: {path}")
        return True
    except Exception as e:
        print(f"[ERRO] Não foi possível abrir {path}: {e}")
        return False

def check_labels_and_distribution(dataset_dir, class_names):
    total_count = 0
    class_counter = Counter()
    for split in ['train', 'val', 'test']:
        print(f"\nVerificando split: {split}")
        for class_name in class_names:
            class_path = os.path.join(dataset_dir, split, class_name)
            if not os.path.exists(class_path):
                print(f"[AVISO] Pasta não encontrada: {class_path}")
                continue
            files = [f for f in os.listdir(class_path) if f.lower().endswith(IMG_EXTS)]
            class_counter[class_name] += len(files)
            total_count += len(files)
            for fname in files:
                img_path = os.path.join(class_path, fname)
                check_image(img_path)
    print("\nResumo de distribuição de classes:")
    for cls in class_names:
        print(f"{cls}: {class_counter[cls]} imagens")
    print(f"Total: {total_count} imagens")

def check_labels_in_code(dataset_dir, class_names):
    """Verifica se os labels atribuídos pelo código batem com as pastas"""
    label_mapping = {name: idx for idx, name in enumerate(class_names)}
    for split in ['train', 'val', 'test']:
        for class_name in class_names:
            class_path = os.path.join(dataset_dir, split, class_name)
            if not os.path.exists(class_path):
                continue
            files = [f for f in os.listdir(class_path) if f.lower().endswith(IMG_EXTS)]
            for fname in files[:5]:  # Verifica só os 5 primeiros de cada pasta
                label = label_mapping.get(class_name)
                if label is None or not isinstance(label, int):
                    print(f"[ERRO] Label inválido para {class_name} ({label})")
                if label < 0 or label >= len(class_names):
                    print(f"[ERRO] Label fora do range para {class_name} ({label})")

if __name__ == "__main__":
    class_names = ['normal', 'covid19', 'pneumonia_viral', 'pneumonia_bacterial']
    print("=== Verificação Inicial do Dataset ===")
    check_labels_and_distribution(DATASET_DIR, class_names)
    check_labels_in_code(DATASET_DIR, class_names)
    print("\nVerificação concluída.")


=== Verificação Inicial do Dataset ===

Verificando split: train

Verificando split: val

Verificando split: test

Resumo de distribuição de classes:
normal: 24060 imagens
covid19: 27041 imagens
pneumonia_viral: 30790 imagens
pneumonia_bacterial: 29500 imagens
Total: 111391 imagens

Verificação concluída.


In [1]:
import os
import time
import gc
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import seaborn as sns
from collections import Counter
from torchvision import models
import torch.nn.functional as F
from tqdm import tqdm

# Configuração de dispositivo
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo utilizado: {device}")

class ChestXRayDataset(Dataset):
    """Dataset personalizado para classificação de pneumonia com cache em RAM"""
    def __init__(self, root_dir, transform=None, cache_in_ram=False):
        self.samples = []
        self.transform = transform
        self.root_dir = root_dir
        self.cache_in_ram = cache_in_ram
        self.cached_images = []
        self.class_names = ['covid19', 'normal', 'pneumonia_bacterial', 'pneumonia_viral']
        label_mapping = {name: idx for idx, name in enumerate(self.class_names)}
        for label_name in os.listdir(root_dir):
            label_dir = os.path.join(root_dir, label_name)
            if not os.path.isdir(label_dir):
                continue
            label = label_mapping.get(label_name.lower())
            if label is None:
                continue
            for img_name in os.listdir(label_dir):
                img_path = os.path.join(label_dir, img_name)
                self.samples.append((img_path, label))

        if self.cache_in_ram:
            print("Carregando imagens na RAM para acelerar o acesso...")
            for img_path, _ in tqdm(self.samples, desc="Cacheando imagens"):
                try:
                    img = Image.open(img_path).convert("RGB")
                except Exception as e:
                    print(f"Erro ao carregar imagem {img_path} durante cache: {e}")
                    img = Image.new('RGB', (224, 224))  # imagem preta substituta
                self.cached_images.append(img)
            print(f"{len(self.cached_images)} imagens carregadas em RAM.")

        print(f"Dataset carregado: {len(self.samples)} amostras de {root_dir}")
        self._print_class_distribution()

    def _print_class_distribution(self):
        labels = [sample[1] for sample in self.samples]
        class_counts = Counter(labels)
        for class_idx, count in class_counts.items():
            print(f"  {self.class_names[class_idx]}: {count} amostras")

    def get_class_weights(self):
        labels = [sample[1] for sample in self.samples]
        class_counts = Counter(labels)
        total_samples = len(self.samples)
        weights = []
        for i in range(len(self.class_names)):
            if i in class_counts:
                weight = total_samples / (len(self.class_names) * class_counts[i])
                weights.append(weight)
            else:
                weights.append(0.0)
        return torch.FloatTensor(weights)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        label = self.samples[idx][1]
        if self.cache_in_ram:
            try:
                image = self.cached_images[idx]
            except IndexError:
                img_path = self.samples[idx][0]
                try:
                    image = Image.open(img_path).convert("RGB")
                except Exception as e:
                    print(f"Erro ao carregar imagem {img_path}: {e}")
                    image = Image.new('RGB', (224, 224))
        else:
            img_path = self.samples[idx][0]
            try:
                image = Image.open(img_path).convert("RGB")
            except Exception as e:
                print(f"Erro ao carregar imagem {img_path}: {e}")
                image = Image.new('RGB', (224, 224))  # imagem preta
        
        if self.transform:
            image = self.transform(image)
        return image, label

    def remove_from_cache(self, idx):
        if self.cache_in_ram and 0 <= idx < len(self.cached_images):
            print(f"Removendo imagem do cache na posição {idx} para liberar RAM")
            self.cached_images[idx] = None

def get_transforms(phase='train'):
    if phase == 'train':
        return transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])
    else:
        return transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])
        ])

class AdvancedDenseNet(nn.Module):
    def __init__(self, num_classes=4, pretrained=True, dropout_rate=0.5):
        super(AdvancedDenseNet, self).__init__()
        self.backbone = models.densenet121(weights='IMAGENET1K_V1' if pretrained else None)
        num_features = self.backbone.classifier.in_features
        self.backbone.classifier = nn.Identity()
        self.classifier = nn.Sequential(
            nn.Dropout(dropout_rate),
            nn.Linear(num_features, 512),
            nn.ReLU(),
            nn.BatchNorm1d(512),
            nn.Dropout(dropout_rate),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )
        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.classifier.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        features = self.backbone.features(x)
        out = F.relu(features, inplace=True)
        out = F.adaptive_avg_pool2d(out, (1, 1)).view(x.size(0), -1)
        return self.classifier(out)

class ModelTrainer:
    def __init__(self, model, train_loader, val_loader, criterion, optimizer,
                 scheduler=None, device='cpu'):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.criterion = criterion
        self.optimizer = optimizer
        self.scheduler = scheduler
        self.device = device
        self.train_losses = []
        self.train_accuracies = []
        self.val_losses = []
        self.val_accuracies = []

    def train(self, num_epochs, early_stopping_patience=10, checkpoint_interval=5):
        best_val_acc = 0.0
        patience_counter = 0
        scaler = torch.cuda.amp.GradScaler(enabled=(self.device.type == 'cuda'))
        print(f"Iniciando treinamento por {num_epochs} épocas...")
        print("-" * 60)
        for epoch in range(num_epochs):
            print(f'Época {epoch+1}/{num_epochs}')
            self.model.train()
            running_loss = 0.0
            correct_predictions = 0
            total_samples = 0
            pbar = tqdm(self.train_loader, desc='Treinamento')
            for images, labels in pbar:
                images, labels = images.to(self.device, non_blocking=True), labels.to(self.device, non_blocking=True)
                with torch.amp.autocast('cuda', enabled=(self.device.type == 'cuda')):
                    outputs = self.model(images)
                    loss = self.criterion(outputs, labels)
                self.optimizer.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(self.optimizer)
                scaler.update()
                running_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                correct_predictions += (predicted == labels).sum().item()
                total_samples += labels.size(0)
                pbar.set_postfix({'loss': loss.item()})
            train_loss = running_loss / len(self.train_loader)
            train_acc = correct_predictions / total_samples
            val_loss, val_acc = self.validate_epoch()
            if self.scheduler:
                self.scheduler.step(val_loss)
            self.train_losses.append(train_loss)
            self.train_accuracies.append(train_acc)
            self.val_losses.append(val_loss)
            self.val_accuracies.append(val_acc)
            print(f'Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}')
            print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f}')
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                patience_counter = 0
                best_model_path = 'best_model.pth'
                if os.path.exists(best_model_path):
                    timestamp = time.strftime("%Y%m%d_%H%M%S")
                    backup_path = f'best_model_backup_{timestamp}.pth'
                    os.rename(best_model_path, backup_path)
                    print(f"Arquivo anterior {best_model_path} renomeado para {backup_path}")
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'val_acc': val_acc,
                    'val_loss': val_loss
                }, best_model_path)
                print(f'Novo melhor modelo salvo! Val Acc: {val_acc:.4f}')
            else:
                patience_counter += 1
            if patience_counter >= early_stopping_patience:
                print(f'Early stopping após {early_stopping_patience} épocas sem melhoria')
                break
            if (epoch + 1) % checkpoint_interval == 0:
                checkpoint_path = f'checkpoint_epoch_{epoch+1}.pth'
                if os.path.exists(checkpoint_path):
                    timestamp = time.strftime("%Y%m%d_%H%M%S")
                    backup_path = f'{checkpoint_path}_backup_{timestamp}'
                    os.rename(checkpoint_path, backup_path)
                    print(f"Checkpoint anterior {checkpoint_path} renomeado para {backup_path}")
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': self.model.state_dict(),
                    'optimizer_state_dict': self.optimizer.state_dict(),
                    'val_acc': val_acc,
                    'val_loss': val_loss
                }, checkpoint_path)
                print(f'Checkpoint salvo na época {epoch+1}')
            print("-" * 60)
        torch.cuda.empty_cache()
        gc.collect()
        print(f'Treinamento concluído! Melhor Val Acc: {best_val_acc:.4f}')
        return best_val_acc

    def validate_epoch(self):
        self.model.eval()
        running_loss = 0.0
        correct_predictions = 0
        total_samples = 0
        pbar = tqdm(self.val_loader, desc='Validação', leave=False)
        with torch.no_grad():
            for images, labels in pbar:
                images, labels = images.to(self.device, non_blocking=True), labels.to(self.device, non_blocking=True)
                with torch.amp.autocast('cuda', enabled=(self.device.type == 'cuda')):
                    outputs = self.model(images)
                    loss = self.criterion(outputs, labels)
                running_loss += loss.item()
                _, predicted = torch.max(outputs, 1)
                correct_predictions += (predicted == labels).sum().item()
                total_samples += labels.size(0)
                pbar.set_postfix({'loss': loss.item()})
        epoch_loss = running_loss / len(self.val_loader)
        epoch_acc = correct_predictions / total_samples
        return epoch_loss, epoch_acc

    def plot_training_history(self):
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
        ax1.plot(self.train_losses, label='Train Loss', color='blue')
        ax1.plot(self.val_losses, label='Validation Loss', color='red')
        ax1.set_title('Curva de Loss')
        ax1.set_xlabel('Época')
        ax1.set_ylabel('Loss')
        ax1.legend()
        ax1.grid(True)
        ax2.plot(self.train_accuracies, label='Train Accuracy', color='blue')
        ax2.plot(self.val_accuracies, label='Validation Accuracy', color='red')
        ax2.set_title('Curva de Acurácia')
        ax2.set_xlabel('Época')
        ax2.set_ylabel('Acurácia')
        ax2.legend()
        ax2.grid(True)
        plt.tight_layout()
        plt.savefig('training_history.png', dpi=300, bbox_inches='tight')
        plt.show()

def evaluate_model(model, test_loader, device, class_names):
    model.eval()
    all_predictions = []
    all_labels = []
    all_probabilities = []
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = model(images)
            probabilities = F.softmax(outputs, dim=1)
            _, predicted = torch.max(outputs, 1)
            all_predictions.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probabilities.extend(probabilities.cpu().numpy())
    print("=== RELATÓRIO DE CLASSIFICAÇÃO ===")
    print(classification_report(all_labels, all_predictions,
                               target_names=class_names, digits=4))
    cm = confusion_matrix(all_labels, all_predictions)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title('Matriz de Confusão')
    plt.ylabel('Rótulo Verdadeiro')
    plt.xlabel('Rótulo Predito')
    plt.savefig('confusion_matrix.png', dpi=300, bbox_inches='tight')
    plt.show()
    all_probabilities = np.array(all_probabilities)
    try:
        auc_scores = {}
        for i, class_name in enumerate(class_names):
            binary_labels = (np.array(all_labels) == i).astype(int)
            auc = roc_auc_score(binary_labels, all_probabilities[:, i])
            auc_scores[class_name] = auc
            print(f"AUC {class_name}: {auc:.4f}")
        mean_auc = np.mean(list(auc_scores.values()))
        print(f"AUC Médio: {mean_auc:.4f}")
    except Exception as e:
        print(f"Erro ao calcular AUC: {e}")
    return all_predictions, all_labels, all_probabilities

def main():
    DATA_DIR = './dataset_organizado'
    BATCH_SIZE = 32
    NUM_EPOCHS = 50
    LEARNING_RATE = 0.001
    WEIGHT_DECAY = 1e-4
    print("=== CNN AVANÇADA PARA CLASSIFICAÇÃO DE PNEUMONIA COM DENSENET-121 (OTIMIZADO) ===")
    print("=" * 60)
    if not os.path.exists(DATA_DIR):
        print(f"ERRO: Diretório {DATA_DIR} não encontrado!")
        print("Por favor, ajuste o caminho DATA_DIR no código.")
        return

    try:
        print("Carregando datasets...")
        train_dataset = ChestXRayDataset(
            os.path.join(DATA_DIR, 'train'),
            transform=get_transforms('train'),
            cache_in_ram=True
        )
        val_dataset = ChestXRayDataset(
            os.path.join(DATA_DIR, 'val'),
            transform=get_transforms('val'),
            cache_in_ram=True
        )
        test_dataset = ChestXRayDataset(
            os.path.join(DATA_DIR, 'test'),
            transform=get_transforms('test'),
            cache_in_ram=True
        )
        class_weights = train_dataset.get_class_weights().to(device)
        print(f"Pesos das classes: {class_weights}")
        train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                                  shuffle=True, num_workers=8, pin_memory=True)
        val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE,
                                shuffle=False, num_workers=8, pin_memory=True)
        test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE,
                                 shuffle=False, num_workers=8, pin_memory=True)
        print("Inicializando modelo...")
        model = AdvancedDenseNet(num_classes=4, pretrained=True, dropout_rate=0.5)
        model = model.to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                                weight_decay=WEIGHT_DECAY)
        scheduler = optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.5, patience=5
        )
        trainer = ModelTrainer(model, train_loader, val_loader, criterion,
                               optimizer, scheduler, device)
        best_val_acc = trainer.train(NUM_EPOCHS, early_stopping_patience=10)
        trainer.plot_training_history()
        if os.path.exists('best_model.pth'):
            print("Carregando melhor modelo para avaliação final...")
            checkpoint = torch.load('best_model.pth', map_location=device)
            model.load_state_dict(checkpoint['model_state_dict'])
        print("\n=== AVALIAÇÃO NO CONJUNTO DE TESTE ===")
        class_names = ['covid19', 'normal', 'pneumonia_bacterial', 'pneumonia_viral']
        predictions, labels, probabilities = evaluate_model(
            model, test_loader, device, class_names
        )
        print("\nTreinamento e avaliação concluídos com sucesso!")
        print(f"Melhor acurácia de validação: {best_val_acc:.4f}")
    except Exception as e:
        print(f"Erro durante a execução: {e}")
        import traceback
        traceback.print_exc()
        print("Limpando cache de imagens em RAM...")
        for ds in [train_dataset, val_dataset, test_dataset]:
            if ds.cache_in_ram:
                for i in range(len(ds.cached_images)):
                    ds.remove_from_cache(i)
        gc.collect()
        torch.cuda.empty_cache()
        print("Memória liberada.")
        print("Verifique se o dataset está organizado corretamente:")
        print("dataset_organizado/")
        print("├── train/")
        print("│   ├── covid19/")
        print("│   ├── normal/")
        print("│   ├── pneumonia_bacterial/")
        print("│   └── pneumonia_viral/")
        print("├── val/")
        print("│   ├── covid19/")
        print("│   ├── normal/")
        print("│   ├── pneumonia_bacterial/")
        print("│   └── pneumonia_viral/")
        print("└── test/")
        print("    ├── covid19/")
        print("    ├── normal/")
        print("    ├── pneumonia_bacterial/")
        print("    └── pneumonia_viral/")

if __name__ == "__main__":
    main()

Dispositivo utilizado: cuda
=== CNN AVANÇADA PARA CLASSIFICAÇÃO DE PNEUMONIA COM DENSENET-121 (OTIMIZADO) ===
Carregando datasets...
Carregando imagens na RAM para acelerar o acesso...


Cacheando imagens: 100%|██████████| 75227/75227 [00:59<00:00, 1268.93it/s]


75227 imagens carregadas em RAM.
Dataset carregado: 75227 amostras de ./dataset_organizado/train
  normal: 15316 amostras
  covid19: 18077 amostras
  pneumonia_viral: 21522 amostras
  pneumonia_bacterial: 20312 amostras
Carregando imagens na RAM para acelerar o acesso...


Cacheando imagens: 100%|██████████| 18090/18090 [00:14<00:00, 1249.63it/s]


18090 imagens carregadas em RAM.
Dataset carregado: 18090 amostras de ./dataset_organizado/val
  normal: 4372 amostras
  covid19: 4491 amostras
  pneumonia_viral: 4631 amostras
  pneumonia_bacterial: 4596 amostras
Carregando imagens na RAM para acelerar o acesso...


...

Train Loss: 0.1163, Train Acc: 0.9549
Val Loss: 0.1453, Val Acc: 0.9443
------------------------------------------------------------
Época 27/50


Treinamento: 100%|██████████| 2351/2351 [03:55<00:00,  9.96it/s, loss=0.224]  


Train Loss: 0.0976, Train Acc: 0.9617
Val Loss: 0.1002, Val Acc: 0.9594
Arquivo anterior best_model.pth renomeado para best_model_backup_20250704_151624.pth
Novo melhor modelo salvo! Val Acc: 0.9594
------------------------------------------------------------
Época 28/50


Treinamento:  33%|███▎      | 777/2351 [01:18<02:39,  9.86it/s, loss=0.135]  


KeyboardInterrupt: 